In [ ]:
# Cell 0: reload Day 8 artifacts
from google.colab import drive; drive.mount('/content/drive')
!pip install -q optuna mlflow lightgbm

import joblib, pandas as pd, numpy as np
PROJECT = "/content/drive/MyDrive/churn-prediction-system"

split = joblib.load(f"{PROJECT}/data/processed/split_v1.joblib")
X_train, X_test = split["X_train"], split["X_test"]
y_train, y_test = split["y_train"], split["y_test"]
numeric, categorical = split["numeric"], split["categorical"]
print("Reloaded:", X_train.shape, "| baseline LightGBM PR-AUC was 0.881")

In [ ]:
# Cell 1: rebuild preprocessor (identical to Day 8)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=20, sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric),
        ("cat", categorical_pipe, categorical),
    ])
print("Preprocessor factory ready.")

In [ ]:
# Cell 2: Optuna study (logs each trial to MLflow)
import optuna, mlflow
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier

mlflow.set_tracking_uri(f"sqlite:///{PROJECT}/mlflow.db")
mlflow.set_experiment("kkbox-churn-active-users")

spw = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":       trial.suggest_int("num_leaves", 20, 200),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "min_child_samples":trial.suggest_int("min_child_samples", 10, 200),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "scale_pos_weight": spw,
        "random_state": 42, "verbose": -1, "n_jobs": -1,
    }
    pipe = Pipeline([("prep", make_preprocessor()), ("model", LGBMClassifier(**params))])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring="average_precision", n_jobs=-1)
    mean_ap = scores.mean()

    # log this trial as an MLflow run
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=False):
        mlflow.set_tag("stage", "optuna_tuning")
        mlflow.log_params({k: v for k, v in params.items() if k != "verbose"})
        mlflow.log_metric("cv_pr_auc", mean_ap)
    return mean_ap

study = optuna.create_study(direction="maximize", study_name="lgbm_pr_auc")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n✅ Tuning complete")
print("Best CV PR-AUC:", round(study.best_value, 4), "(baseline was 0.881)")
print("Best params:", study.best_params)

In [ ]:
# Reload baseline model + split, then diagnose
import joblib, numpy as np, pandas as pd
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, brier_score_loss

PROJECT = "/content/drive/MyDrive/churn-prediction-system"
split = joblib.load(f"{PROJECT}/data/processed/split_v1.joblib")
X_train, X_test = split["X_train"], split["X_test"]
y_train, y_test = split["y_train"], split["y_test"]

pipe = joblib.load(f"{PROJECT}/models/baseline_lightgbm.joblib")
proba = pipe.predict_proba(X_test)[:, 1]

print("=== Test-set metrics ===")
print("Log loss:  ", round(log_loss(y_test, proba), 4), "  (winner: 0.0794, full population)")
print("ROC-AUC:   ", round(roc_auc_score(y_test, proba), 4))
print("PR-AUC:    ", round(average_precision_score(y_test, proba), 4))
print("Brier:     ", round(brier_score_loss(y_test, proba), 4))

# --- Feature importance, mapped to readable names ---
model = pipe.named_steps["model"]
prep  = pipe.named_steps["prep"]
feat_names = prep.get_feature_names_out()

imp = pd.DataFrame({
    "feature": feat_names,
    "gain": model.booster_.feature_importance(importance_type="gain")
})
imp["pct"] = 100 * imp["gain"] / imp["gain"].sum()
imp = imp.sort_values("pct", ascending=False).reset_index(drop=True)

print("\n=== Top 15 features by gain contribution ===")
print(imp.head(15).to_string(index=False))

print("\n=== Concentration check ===")
print(f"Top 1 feature:  {imp.iloc[0]['pct']:.1f}%")
print(f"Top 2 features: {imp.head(2)['pct'].sum():.1f}%")
print(f"Top 5 features: {imp.head(5)['pct'].sum():.1f}%")
print(f"Features needed to reach 90% of gain: "
      f"{(imp['pct'].cumsum() <= 90).sum() + 1}")
print(f"\nLog-feature total contribution: "
      f"{imp[imp['feature'].str.contains('secs|unq|plays|active_days|completion|has_logs')]['pct'].sum():.1f}%")

In [ ]:
# Refit best Optuna config on full train, evaluate on the locked test set
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

best = study.best_params
best.update({"scale_pos_weight": spw, "random_state": 42, "verbose": -1, "n_jobs": -1})

tuned = Pipeline([("prep", make_preprocessor()), ("model", LGBMClassifier(**best))])
tuned.fit(X_train, y_train)
proba = tuned.predict_proba(X_test)[:, 1]

print("TUNED on test  → ROC-AUC:", round(roc_auc_score(y_test, proba), 4),
      "| PR-AUC:", round(average_precision_score(y_test, proba), 4))
print("BASELINE test  → ROC-AUC: 0.9803 | PR-AUC: 0.8813")

# Save the tuned model + best params
import joblib, json
joblib.dump(tuned, f"{PROJECT}/models/tuned_lightgbm.joblib")
json.dump(best, open(f"{PROJECT}/models/best_params.json", "w"), indent=2)
print("\n✅ Saved tuned model + params")

## Summary — Hyperparameter Tuning with Optuna

**GridSearchCV vs Optuna (plain terms).**
- *GridSearchCV* tries every combination on a fixed grid — exhaustive and dumb. 5 values across 5 hyperparameters = 3,125 fits, most of them wasted on bad regions.
- *Optuna* treats tuning as an optimisation problem: it learns from each trial which regions look promising and concentrates its budget there (Bayesian-style search), and prunes hopeless trials early. It explores a *continuous* space intelligently in far fewer trials.

We used **Optuna** to reflect current tooling and because it searches continuous ranges (learning rate, regularisation) that a grid can't cover efficiently.

**What we did.** 30 trials, each scored by 3-fold stratified cross-validation on the *training set only* (test set never touched — tuning against it would leak). PR-AUC was the optimisation target. Every trial logged to MLflow (clearing the 15+ tracked-runs goal).

**Result — a deliberate finding, not a win.** Tuning did **not** beat the baseline: best CV PR-AUC ≈ 0.878 vs the baseline's ~0.881, statistically indistinguishable. The trial scores plateaued (0.867–0.878) regardless of configuration, with the best configs favouring low learning rate + heavy regularisation ("go slow and careful").

**Interpretation:** the predictive ceiling here is set by the *features*, not hyperparameters — extracting nearly all available signal happens with default-ish settings. This pointed us toward richer feature engineering (Notebook 03) rather than more tuning. Even though the final production model later changed, performing the Optuna study established this ceiling rigorously and demonstrates the modern tuning workflow.